# Differentiable No-Arbitrage Projection Layer (M2) Validation
This notebook validates the calibration stability, speed benchmarks (loop-free vectorized vs sequential loop), and static no-arbitrage checks for Phase 9.

In [1]:
import sys
import os
import time
import numpy as np
import torch

# Ensure deepvol package is on path
sys.path.insert(0, os.path.abspath("../src"))

from deepvol.arbitrage.projection_layer import DifferentiableArbitrageFreeProjection, bs_call_price_pt, bs_iv_inversion_hybrid
from deepvol.mrm.arbitrage import check_arbitrage

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.8.0+cu128
CUDA available: True


In [2]:
class SequentialProjection(DifferentiableArbitrageFreeProjection):
    def forward(self, iv_surface: torch.Tensor) -> torch.Tensor:
        orig_dtype = iv_surface.dtype
        device = iv_surface.device
        iv_double = iv_surface.to(torch.float64)
        B, nT, nK = iv_double.shape
        T_m = self.T_grid.view(1, nT, 1).expand(B, nT, nK).to(device)
        K_m = self.K_abs.view(1, 1, nK).expand(B, nT, nK).to(device)
        S0_tensor = torch.full_like(iv_double, self.S0, device=device)
        iv_clamped = torch.clamp(iv_double, min=1e-4, max=5.0)
        C = bs_call_price_pt(S0_tensor, K_m, T_m, iv_clamped)
        
        C_proj = C.clone()
        C_proj[:, :, -1] = torch.clamp(C_proj[:, :, -1], min=0.0)
        h = K_m[:, :, 1:] - K_m[:, :, :-1]
        d = (C_proj[:, :, 1:] - C_proj[:, :, :-1]) / h
        d_clamped = torch.clamp(d, max=0.0)
        d_conv = torch.cummax(d_clamped, dim=2)[0]
        
        # OLD sequential Python loop
        for j in range(nK - 2, -1, -1):
            C_proj[:, :, j] = C_proj[:, :, j+1] - d_conv[:, :, j] * h[:, :, j]
            
        intrinsic = torch.clamp(S0_tensor - K_m, min=0.0)
        C_proj = torch.max(C_proj, intrinsic)
        C_proj = torch.where(C_proj - intrinsic < 1e-7 * S0_tensor, intrinsic, C_proj)
        
        iv_proj = bs_iv_inversion_hybrid(C_proj, S0_tensor, K_m, T_m)
        W = iv_proj**2 * T_m
        W_proj = torch.cummax(W, dim=1)[0]
        T_m_safe = torch.clamp(T_m, min=1e-12)
        iv_final = torch.sqrt(W_proj / T_m_safe)
        return torch.clamp(iv_final, min=1e-4, max=5.0).to(orig_dtype)


In [3]:
# Setup test grids
T_grid = np.linspace(0.1, 2.0, 20)
K_grid = np.linspace(-0.5, 0.5, 100)
B = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running benchmarks on: {device}")

iv_raw = np.full((B, len(T_grid), len(K_grid)), 0.3)
iv_raw[:, 5, :] = 0.05
iv_raw[:, :, 50] = 0.01
iv_tensor = torch.tensor(iv_raw, dtype=torch.float64, device=device)

# Instantiate projection layers
proj_loop = SequentialProjection(T_grid, K_grid, S0=1.0).to(device)
proj_vectorized = DifferentiableArbitrageFreeProjection(T_grid, K_grid, S0=1.0).to(device)

# Warmup
for _ in range(5):
    _ = proj_loop(iv_tensor)
    _ = proj_vectorized(iv_tensor)

if device == "cuda":
    torch.cuda.synchronize()

# Benchmark Sequential Python Loop
t0 = time.perf_counter()
n_runs = 50
for _ in range(n_runs):
    _ = proj_loop(iv_tensor)
if device == "cuda":
    torch.cuda.synchronize()
t_loop = (time.perf_counter() - t0) / n_runs

# Benchmark SOTA Loop-Free Vectorized Cumsum
t0 = time.perf_counter()
for _ in range(n_runs):
    _ = proj_vectorized(iv_tensor)
if device == "cuda":
    torch.cuda.synchronize()
t_vectorized = (time.perf_counter() - t0) / n_runs

print(f"Sequential Python Loop Average Execution Time: {t_loop * 1000:.3f} ms")
print(f"SOTA Loop-free Vectorized Average Execution Time: {t_vectorized * 1000:.3f} ms")
print(f"Speedup: {t_loop / t_vectorized:.2f}x")


Running benchmarks on: cuda


Sequential Python Loop Average Execution Time: 10.612 ms
SOTA Loop-free Vectorized Average Execution Time: 6.271 ms
Speedup: 1.69x


In [4]:
# Setup arbitrage verification
T_grid_arb = np.array([0.1, 0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.0])
K_grid_arb = np.array([-0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5])
nT = len(T_grid_arb)
nK = len(K_grid_arb)
B_arb = 2

iv_raw_arb = np.full((B_arb, nT, nK), 0.3)
iv_raw_arb[:, 2, :] = 0.05
iv_raw_arb[:, 0, 5] = 0.01

print("Before projection arbitrage status:")
for b in range(B_arb):
    res_before = check_arbitrage(iv_raw_arb[b], K_grid_arb, T_grid_arb, S=1.0)
    print(f"Batch {b} Has Calendar Arbitrage:", res_before["calendar"]["has_arbitrage"])
    print(f"Batch {b} Has Durrleman Butterfly Arbitrage:", res_before["butterfly_durrleman"]["has_arbitrage"])
    print(f"Batch {b} Has Price Butterfly Arbitrage:", res_before["butterfly_price"]["has_arbitrage"])

# Project
iv_tensor_arb = torch.tensor(iv_raw_arb, dtype=torch.float64, device=device)
projection_arb = DifferentiableArbitrageFreeProjection(T_grid_arb, K_grid_arb, S0=1.0).to(device)
iv_projected_arb = projection_arb(iv_tensor_arb)

# Verify
iv_proj_arb_np = iv_projected_arb.detach().cpu().numpy()
print("\nAfter projection arbitrage status:")
for b in range(B_arb):
    res_after = check_arbitrage(iv_proj_arb_np[b], K_grid_arb, T_grid_arb, S=1.0)
    print(f"Batch {b} Has Calendar Arbitrage:", res_after["calendar"]["has_arbitrage"])
    print(f"Batch {b} Has Durrleman Butterfly Arbitrage:", res_after["butterfly_durrleman"]["has_arbitrage"])
    print(f"Batch {b} Has Price Butterfly Arbitrage:", res_after["butterfly_price"]["has_arbitrage"])
    
    assert not res_after["calendar"]["has_arbitrage"]
    assert not res_after["butterfly_durrleman"]["has_arbitrage"]
    assert not res_after["butterfly_price"]["has_arbitrage"]

print("\nAll arbitrage checks passed successfully! 100% of calendar and butterfly arbitrage violations are resolved.")


Before projection arbitrage status:
Batch 0 Has Calendar Arbitrage: True
Batch 0 Has Durrleman Butterfly Arbitrage: False
Batch 0 Has Price Butterfly Arbitrage: True
Batch 1 Has Calendar Arbitrage: True
Batch 1 Has Durrleman Butterfly Arbitrage: False
Batch 1 Has Price Butterfly Arbitrage: True

After projection arbitrage status:
Batch 0 Has Calendar Arbitrage: False
Batch 0 Has Durrleman Butterfly Arbitrage: False
Batch 0 Has Price Butterfly Arbitrage: False
Batch 1 Has Calendar Arbitrage: False
Batch 1 Has Durrleman Butterfly Arbitrage: False
Batch 1 Has Price Butterfly Arbitrage: False

All arbitrage checks passed successfully! 100% of calendar and butterfly arbitrage violations are resolved.


In [5]:
# Setup gradient verification
iv_raw_grad = torch.tensor(iv_raw_arb, dtype=torch.float64, device=device, requires_grad=True)
projection_grad = DifferentiableArbitrageFreeProjection(T_grid_arb, K_grid_arb, S0=1.0).to(device)

iv_projected_grad = projection_grad(iv_raw_grad)

loss = iv_projected_grad.sum()
loss.backward()

print("Is gradient populated:", iv_raw_grad.grad is not None)
print("Are all gradients finite:", torch.all(torch.isfinite(iv_raw_grad.grad)).item())
print("Gradient shape:", iv_raw_grad.grad.shape)
print("Mean gradient magnitude:", torch.abs(iv_raw_grad.grad).mean().item())

# Assertions
assert iv_raw_grad.grad is not None
assert torch.all(torch.isfinite(iv_raw_grad.grad))
print("\nGradient flow and differentiability verified successfully!")


Is gradient populated: True
Are all gradients finite: True
Gradient shape: torch.Size([2, 8, 11])
Mean gradient magnitude: 0.8383883476483638

Gradient flow and differentiability verified successfully!
